# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [70]:
import numpy as np

# Estamos modelando el mundo de la imagen. Creamos la funcion de probabilidad de movimiento del robot en el almacén.
class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Posición inicial
        self.start = (0, 0)

        # Estanterías / paredes -> No cuentan como estados transitables
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2)
        }

        # Celdas de piso resbaloso
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3)
        }

        # Estados terminales y su recompensa
        self.terminal_states = {
            (0, 5): 10,    # Entrega
            (2, 2): 2,     # Carga
            (3, 5): -10    # Peligro mortal
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 4): -3,
            (4, 1): -3
        }

        # Costo por paso
        self.living_reward = -1.0

        # Factor de descuento
        self.gamma = 0.9

        # Acciones
        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state

        # Debe estar dentro del grid
        if not (0 <= row < self.height and
                0 <= col < self.width):
            return False

        # No puede ser una estantería
        if state in self.walls:
            return False

        return True

    def states(self):
        # Todos los estados transitables
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # Primero revisamos si es terminal
        if state in self.terminal_states:
            return self.terminal_states[state]

        # Luego peligros no terminales
        if state in self.danger_states:
            return self.danger_states[state]

        # Cualquier otro estado transitable
        return self.living_reward

# Función de transición del robot en el almacén. Calcula las probabilidades de moverse a los estados vecinos considerando el tipo de piso.
    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Las probabilidades dependen del tipo de piso.
        Si el movimiento sale del grid o golpea una pared,
        el robot permanece en el mismo estado.
        """

        # Los terminales no tienen movimientos relevantes
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades según el piso
        if state in self.slippery_states:
            p_forward = 0.60
            p_left = 0.20
            p_right = 0.20
        else:
            p_forward = 0.90
            p_left = 0.05
            p_right = 0.05

        # Dirección elegida
        dr, dc = action

        # Direcciones de desviación
        # Rotación izquierda: (dr, dc) -> (dc, -dr)
        left_action = (dc, -dr)

        # Rotación derecha: (dr, dc) -> (-dc, dr)
        right_action = (-dc, dr)

        movements = [
            (action, p_forward),
            (left_action, p_left),
            (right_action, p_right)
        ]

        transition_probs = {}

        for movement, probability in movements:
            next_state = (
                state[0] + movement[0],
                state[1] + movement[1]
            )

            # Si choca contra pared o sale del grid,
            # permanece en el mismo estado
            if not self.is_valid_state(next_state):
                next_state = state

            # Acumulamos probabilidades si varios movimientos
            # terminan en el mismo estado
            transition_probs[next_state] = (
                transition_probs.get(next_state, 0)
                + probability
            )

        return list(transition_probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [71]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [72]:
def expected_next_value(grid, state, action, V):

    total = 0.0

    for next_state, probability in grid.get_transition_probs(state, action):
        total += probability * V[next_state]

    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):

    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):

        V_new = {}

        for state in grid.states():

            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
                continue

            action_values = []

            for action in grid.actions:
                expected_value = expected_next_value(
                    grid, state, action, V
                )

                action_values.append(expected_value)

            best_expected_value = max(action_values)

            V_new[state] = (
                grid.get_reward(state)
                + grid.gamma * best_expected_value
            )

        delta = max(
            abs(V_new[state] - V[state])
            for state in grid.states()
        )

        V = V_new

        if delta < threshold:
            return V, iteration + 1

    return V, max_iter


def extract_policy(grid, V):

    policy = {}

    for state in grid.states():

        # Los terminales no necesitan una acción
        if grid.is_terminal(state):
            policy[state] = None
            continue

        best_action = None
        best_value = -np.inf

        for action in grid.actions:

            value = expected_next_value(
                grid,
                state,
                action,
                V
            )

            if value > best_value:
                best_value = value
                best_action = action

        policy[state] = best_action

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [73]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # TODO
    pass


def policy_improvement(grid, V):
    # TODO
    pass


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    # 1. política inicial arbitraria
    # 2. evaluación
    # 3. mejora
    # 4. repetir hasta estabilidad
    pass



## Parte 4 — Visualización y comparación


In [74]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [75]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


TypeError: cannot unpack non-iterable NoneType object


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

Desde START, la política óptima va por la carga +2, no por la entrega +10. La primera acción es avanzar a la derecha y prioriza la ruta hacia la zona de +2. Tiene que ver esto con el precio de vivir (-1.0)

2. ¿Por qué una recompensa menor podría ser óptima?

Una recompensa menor puede ser óptima cuando su retorno esperado es mayor: menos riesgo de caer en estados muy negativos, menos pasos acumulando costo, y menor penalización por incertidumbre y descuento.

3. ¿En qué estados el piso resbaloso cambia la decisión?

El piso resbaloso cambia decisiones sobre todo en los estados resbalosos y su vecindad, porque aumenta la probabilidad de desviarse. En este mapa, eso afecta especialmente la zona de (1,2), (2,1) y (3,3), donde la política tiende a evitar maniobras que te puedan desviar hacia peligro o rutas largas.

4. ¿Qué papel cumple el costo por paso `-1`?

El costo por paso -1 presiona a terminar rápido: penaliza vueltas, favorece trayectorias cortas y puede hacer preferible una meta menor pero cercana si la meta grande implica mucho riesgo o demora.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

T(s,a,s') ya no puede usar las mismas probabilidades en todos los estados porque la dinámica depende del estado actual: en piso normal y resbaloso las probabilidades son distintas, y además paredes/bordes hacen que parte de la probabilidad se convierta en quedarse en el mismo estado.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

Resultados: Al haber menor precio de vivir, el agente se va directamente por la salida +10 pues no necesita un punto de checkpoint que le devuelva cierta cantidad de puntos.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

Resultado: empeora el control en piso resbaloso, pero no lo suficiente para cambiar la mejor decisión global.

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

Resultado: hace al agente más paciente, así que valora más la recompensa grande aunque esté más lejos y tenga más riesgo.

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

RESULTADO: La política deja de priorizar ir a +2 cuando baja mas de -0.8 masomenos